In [ ]:
import pymysql
import pandas as pd
import numpy as np

# Load feature-engineered dataset
df = pd.read_csv(r"C:\Users\Vanitha\OneDrive\Documents\GUVI\capstone_project\job_acceptance_prediction_system\data\cleaned\job_acceptance_f_e_data.csv")

# Quick check
df.info() 
df.head() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51500 entries, 0 to 51499
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age_years                    51500 non-null  float64
 1   gender                       51500 non-null  int64  
 2   ssc_percentage               51500 non-null  float64
 3   hsc_percentage               51500 non-null  float64
 4   degree_percentage            51500 non-null  float64
 5   degree_specialization        51500 non-null  int64  
 6   technical_score              51500 non-null  float64
 7   aptitude_score               51500 non-null  float64
 8   communication_score          51500 non-null  float64
 9   skills_match_percentage      51500 non-null  float64
 10  certifications_count         51500 non-null  float64
 11  internship_experience        51500 non-null  int64  
 12  years_of_experience          51500 non-null  float64
 13  career_switch_wi

,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,relocation_willingness,status,academic_band,experience_category,skills_match_level,interview_score_avg,interview_performance,placement_probability_score,ctc_gap,ctc_gap_flag
0,-0.125568,1,-0.635376,1.291098,0.931394,0,-0.827125,2.576075,-0.164746,0.475780,...,0,0,Low,Fresher,Low,0.528068,Poor,0.004758,-0.151237,0
1,-0.870846,1,-0.272108,-1.169454,-0.963692,1,0.329015,-0.948746,-0.507640,-0.052684,...,0,0,Low,Fresher,NaN,-0.375790,Poor,-0.000527,-1.477382,0
2,1.364987,0,0.500596,-0.243202,2.051606,2,0.372630,-0.546106,1.351319,0.129682,...,0,1,Low,Fresher,Low,0.392614,Poor,0.001297,-1.328537,0
3,0.868135,1,0.533154,0.121118,-0.406301,3,0.913382,-0.300701,-1.248190,-0.022133,...,1,0,Low,Fresher,NaN,-0.211836,Poor,-0.000221,1.885968,0
4,0.122857,1,-1.225270,-1.493163,0.395154,2,0.021871,0.172139,-0.471201,1.276681,...,1,0,Low,Fresher,Low,-0.092397,Poor,0.012767,-1.201500,0


In [2]:
# MySQL cannot handle NaN, so replace them with None
df = df.replace({np.nan: None}) 

In [3]:
# Connect to MySQL
conn = pymysql.connect(
    host='localhost',     
    user='root',           
    password='Vani@gupa261991',  
    port=3306,
    charset='utf8mb4'
)

cursor = conn.cursor() 

cursor.execute("CREATE DATABASE IF NOT EXISTS job_placement_db;")
cursor.execute("USE job_placement_db;") 
print("Database created")

Database created


In [4]:
create_table_query = """
CREATE TABLE IF NOT EXISTS candidate_data (
    candidate_id INT AUTO_INCREMENT PRIMARY KEY,
    age_years INT,
    gender VARCHAR(10),
    ssc_percentage FLOAT,
    hsc_percentage FLOAT,
    degree_percentage FLOAT,
    degree_specialization VARCHAR(50),
    internship_experience VARCHAR(10),
    technical_score FLOAT,
    aptitude_score FLOAT,
    communication_score FLOAT,
    skills_match_percentage FLOAT,
    certifications_count INT,
    years_of_experience FLOAT,
    career_switch_willingness VARCHAR(20),
    relevant_experience VARCHAR(20),
    previous_ctc_lpa FLOAT,
    expected_ctc_lpa FLOAT,
    company_tier INT,
    job_role_match VARCHAR(20),
    competition_level VARCHAR(20),
    bond_requirement VARCHAR(20),
    notice_period_days INT,
    layoff_history VARCHAR(10),
    employment_gap_months INT,
    relocation_willingness VARCHAR(20),
    status VARCHAR(20),
    interview_performance INT,
    experience_category INT,
    skills_match_level INT,
    academic_band INT,
    placement_probability_score DOUBLE
);
"""
cursor.execute(create_table_query)
conn.commit()
print("Table Created")

Table Created


In [5]:
# Inspect Problem Columns
df[['academic_band', 'skills_match_level']].head() 

,academic_band,skills_match_level
0,Low,Low
1,Low,None
2,Low,Low
3,Low,None
4,Low,Low


In [ ]:
# MAP STRING CATEGORIES TO NUMERIC

# Experience category
experience_map = {
    "Fresher": 0,
    "Junior": 1,
    "Experienced": 2
}

if 'experience_category' in df.columns:
    df['experience_category'] = df['experience_category'].map(experience_map)

# Level-based categories
level_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

if 'skills_match_level' in df.columns:
    df['skills_match_level'] = df['skills_match_level'].map(level_map)

if 'academic_band' in df.columns:
    df['academic_band'] = df['academic_band'].map(level_map)

# Interview performance mapping
interview_map = {
    "Poor": 0,
    "Average": 1,
    "Good": 2
}

if 'interview_performance' in df.columns:
    df['interview_performance'] = df['interview_performance'].map(interview_map) 

In [ ]:
# Replace NaN with MySQL
df = df.replace({np.nan: None}) 

In [8]:
df[['experience_category','skills_match_level','academic_band','interview_performance']].head() 

,experience_category,skills_match_level,academic_band,interview_performance
0,0,0.0,0,0
1,0,None,0,0
2,0,0.0,0,0
3,0,None,0,0
4,0,0.0,0,0


In [9]:
# Align DataFrame to Table Columns
cursor.execute("DESCRIBE candidate_data;")
table_cols = [row[0] for row in cursor.fetchall() if row[0] != "candidate_id"]

df_sql = df[[col for col in df.columns if col in table_cols]]

df_sql = df_sql.where(pd.notnull(df_sql), None) 


In [10]:
cursor.execute("TRUNCATE TABLE candidate_data;")
conn.commit()

In [11]:
# Prepare insert query
cols = ",".join(df_sql.columns)
placeholders = ",".join(["%s"] * len(df_sql.columns))
insert_query = f"INSERT INTO candidate_data ({cols}) VALUES ({placeholders})"

# Convert DataFrame to list of tuples
data = [tuple(x) for x in df_sql.to_numpy()] 

In [12]:
# Insert Data in chunks
chunk_size = 5000

for i in range(0, len(data), chunk_size):
    chunk = data[i:i + chunk_size]
    cursor.executemany(insert_query, chunk)
    conn.commit()
    print(f"Inserted rows {i} to {i + len(chunk) - 1}")


Inserted rows 0 to 4999
Inserted rows 5000 to 9999
Inserted rows 10000 to 14999
Inserted rows 15000 to 19999
Inserted rows 20000 to 24999
Inserted rows 25000 to 29999
Inserted rows 30000 to 34999
Inserted rows 35000 to 39999
Inserted rows 40000 to 44999
Inserted rows 45000 to 49999
Inserted rows 50000 to 51499


In [13]:
# Verify & Close connection
cursor.execute("SELECT COUNT(*) FROM candidate_data;")
print("Total rows inserted:", cursor.fetchone()[0])

cursor.close()
conn.close()
print("MySQL connection closed successfully.") 

Total rows inserted: 51500
MySQL connection closed successfully.
